In [62]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

def clean_form(raw_form):
    if not isinstance(raw_form, str):
        return ""

    tokens = raw_form.split()
    cleaned = []

    for t in tokens:
        t = t.upper()
        if t.startswith("W"):
            cleaned.append("W")
        elif t.startswith("D"):
            cleaned.append("D")
        elif t.startswith("L"):
            cleaned.append("L")

    return " ".join(cleaned)


def scrape_bbc_standings():
    url = "https://www.bbc.com/sport/football/premier-league/table"
    headers = {"User-Agent": "Mozilla/5.0"}

    print(f"[✓] Requesting {url} ...")
    response = requests.get(url, headers=headers)

    if response.status_code != 200:
        print("Request failed:", response.status_code)
        return None

    print("[✓] Status:", response.status_code)

    soup = BeautifulSoup(response.content, "html.parser")
    table = soup.find("table")

    if not table:
        print("No table found. BBC changed structure.")
        return None

    rows = table.find_all("tr")

    # getting the header row
    header = [th.get_text(" ", strip=True) for th in rows[0].find_all("th")]

    # gathering all rows
    data = []
    for row in rows[1:]:
        cols = [td.get_text(" ", strip=True) for td in row.find_all("td")]
        if len(cols) == len(header):   # ensure row matches header length
            data.append(cols)

    # Build dataframe
    df = pd.DataFrame(data, columns=header)

    # Fix the form column
    form_col = [c for c in df.columns if "Form" in c]

    if form_col:
        col = form_col[0]                     # actual BBC form column name
        df["Form"] = df[col].apply(clean_form)
        df.drop(columns=[col], inplace=True)  # remove messy BBC column

    print("\n============================================================")
    print("Premier League Standings (Cleaned Form)")
    print("============================================================")
    print(df)

    return df


print("\nScraping Premier League Data...\n")
scrape_bbc_standings()
print("\nScraping Complete!\n")



Scraping Premier League Data...

[✓] Requesting https://www.bbc.com/sport/football/premier-league/table ...
[✓] Status: 200

Premier League Standings (Cleaned Form)
                          Team Played Won Drawn Lost Goals For Goals Against  \
0                    1 Arsenal     12   9     2    1        24             6   
1                    2 Chelsea     12   7     2    3        23            11   
2            3 Manchester City     12   7     1    4        24            10   
3                4 Aston Villa     12   6     3    3        15            11   
4             5 Crystal Palace     12   5     5    2        16             9   
5     6 Brighton & Hove Albion     12   5     4    3        19            16   
6                 7 Sunderland     12   5     4    3        14            11   
7            8 AFC Bournemouth     12   5     4    3        19            20   
8          9 Tottenham Hotspur     12   5     3    4        20            14   
9         10 Manchester United    